# Pandas Cleaning, GroupBy, and Joining
Real data is usually untidy long before any model sees it. Pandas shines in the messy middle where columns need cleaning, groups need summarizing, and separate tables need to be joined back into one usable view.

Cleaning work is not glamorous, but it is where analytical trust is built. A model trained on badly merged, silently missing, or incorrectly typed data can look sophisticated while being completely unreliable.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'team': ['A', 'A', 'B', 'B', 'C'],
    'sales': [100, None, 150, 130, None],
    'quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1']
})
print(df)
print(df.isna().sum())

  team  sales quarter
0    A  100.0      Q1
1    A    NaN      Q2
2    B  150.0      Q1
3    B  130.0      Q2
4    C    NaN      Q1
team       0
sales      2
quarter    0
dtype: int64


In [2]:
clean = df.copy()
clean['sales_filled_with_mean'] = clean['sales'].fillna(clean['sales'].mean())
clean['sales_filled_with_zero'] = clean['sales'].fillna(0)
print(clean)

  team  sales quarter  sales_filled_with_mean  sales_filled_with_zero
0    A  100.0      Q1              100.000000                   100.0
1    A    NaN      Q2              126.666667                     0.0
2    B  150.0      Q1              150.000000                   150.0
3    B  130.0      Q2              130.000000                   130.0
4    C    NaN      Q1              126.666667                     0.0


Different fill strategies carry different meanings. Replacing with zero says something very different from replacing with a group mean or dropping the row entirely. Data cleaning is never just syntax. It is also a modeling assumption.

In [3]:
grouped = clean.groupby('team', dropna=False)['sales_filled_with_mean'].agg(['count', 'mean', 'sum'])
print(grouped)

      count        mean         sum
team                               
A         2  113.333333  226.666667
B         2  140.000000  280.000000
C         1  126.666667  126.666667


A groupby call is best understood as a split-apply-combine process. Pandas partitions rows by keys, applies an operation inside each group, and then combines the results into a new object. That mental model helps you predict the output shape.

In [4]:
left = pd.DataFrame({'team': ['A', 'B', 'C'], 'manager': ['Nina', 'Omar', 'Priya']})
right = pd.DataFrame({'team': ['A', 'B', 'D'], 'city': ['Hyd', 'Blr', 'Chn']})
print('inner join:\n', left.merge(right, on='team', how='inner'))
print('left join:\n', left.merge(right, on='team', how='left'))

inner join:
   team manager city
0    A    Nina  Hyd
1    B    Omar  Blr
left join:
   team manager city
0    A    Nina  Hyd
1    B    Omar  Blr
2    C   Priya  NaN


Joins deserve close reading because they are one of the easiest places to multiply rows unexpectedly or lose records silently. Before trusting a merge, it is worth asking what the key cardinality is supposed to be.

In [5]:
wide = pd.DataFrame({
    'id': [1, 2],
    'math': [90, 80],
    'science': [85, 88]
})
long = wide.melt(id_vars='id', var_name='subject', value_name='score')
print(long)

   id  subject  score
0   1     math     90
1   2     math     80
2   1  science     85
3   2  science     88


Reshaping is another underappreciated skill. Many plotting, feature engineering, and reporting tasks become easier once you are comfortable moving between wide and long layouts.

In [6]:
typed = pd.DataFrame({'value': ['10', '20', 'bad', '40']})
typed['numeric'] = pd.to_numeric(typed['value'], errors='coerce')
print(typed)

  value  numeric
0    10     10.0
1    20     20.0
2   bad      NaN
3    40     40.0


Strong pandas intuition comes from combining syntax with data-meaning questions: what does missing mean, what should the key uniqueness be, what granularity should this table represent, and which rows am I willing to drop or duplicate?

## Final Takeaway
Cleaning, grouping, and joining are where pandas becomes genuinely useful for real projects. The deep skill is not remembering method names alone. It is understanding the data meaning behind fills, aggregations, merge cardinality, and reshaping choices.